# Importer Walkthrough

This notebook shows the importer layer in isolation: tiny stand-in Scryfall
and MTGJSON payloads go in, normalized catalog and price rows come out.


## Setup

The importer notebook is self-contained. It writes tiny JSON fixtures into a
scratch workspace, migrates a fresh database, and then runs the same import
functions the CLI uses.


In [ ]:
from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'mtg_source_stack').exists():
            return candidate
    raise RuntimeError('Could not find the repo root from the current working directory.')


REPO_ROOT = find_repo_root(Path.cwd())
SRC_DIR = REPO_ROOT / 'src'
SRC_DIR_STR = str(SRC_DIR)
sys.path = [path for path in sys.path if path != SRC_DIR_STR]
sys.path.insert(0, SRC_DIR_STR)

WORK_DIR = REPO_ROOT / 'var' / 'notebooks' / '02_importer'
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True)

from mtg_source_stack.db.connection import connect
from mtg_source_stack.db.schema import initialize_database
from mtg_source_stack.importer.mtgjson import import_mtgjson_identifiers, import_mtgjson_prices
from mtg_source_stack.importer.scryfall import import_scryfall_cards
from mtg_source_stack.inventory.report_helpers import render_table

DB_PATH = WORK_DIR / 'importer_walkthrough.sqlite3'
SCRYFALL_JSON = WORK_DIR / 'scryfall_sample.json'
IDENTIFIERS_JSON = WORK_DIR / 'identifiers_sample.json'
PRICES_JSON = WORK_DIR / 'prices_sample.json'

print('repo root:', REPO_ROOT)
print('work dir:', WORK_DIR)
print('db path:', DB_PATH)


## Build Tiny Sample Payloads

The sample data mirrors the real importer shape but stays small enough to
read in one pass. It includes one non-USD MTGJSON row on purpose so the
later price table can show that the runtime import keeps USD snapshots only.


In [ ]:
scryfall_payload = [
    {
        'id': 's1',
        'oracle_id': 'o1',
        'name': 'Lightning Bolt',
        'set': 'lea',
        'set_name': 'Limited Edition Alpha',
        'collector_number': '161',
        'lang': 'en',
        'rarity': 'common',
        'released_at': '1993-08-05',
        'mana_cost': '{R}',
        'type_line': 'Instant',
        'oracle_text': 'Lightning Bolt deals 3 damage to any target.',
        'colors': ['R'],
        'color_identity': ['R'],
        'finishes': ['nonfoil'],
        'legalities': {'commander': 'legal'},
        'purchase_uris': {'tcgplayer': 'https://example.test/tcg/lightning-bolt'},
        'tcgplayer_id': 534658,
        'cardmarket_id': 752712,
    },
    {
        'id': 's2',
        'oracle_id': 'o2',
        'name': 'Sol Ring',
        'set': 'clb',
        'set_name': "Commander Legends: Battle for Baldur's Gate",
        'collector_number': '860',
        'lang': 'en',
        'rarity': 'rare',
        'released_at': '2022-06-10',
        'mana_cost': '{1}',
        'type_line': 'Artifact',
        'oracle_text': '{T}: Add {C}{C}.',
        'colors': [],
        'color_identity': [],
        'finishes': ['nonfoil', 'foil'],
        'legalities': {'commander': 'legal'},
        'purchase_uris': {'tcgplayer': 'https://example.test/tcg/sol-ring'},
        'tcgplayer_id': 271111,
        'cardmarket_id': 665555,
    },
    {
        'id': 's3',
        'oracle_id': 'o3',
        'name': 'Counterspell',
        'set': '7ed',
        'set_name': 'Seventh Edition',
        'collector_number': '67',
        'lang': 'en',
        'rarity': 'uncommon',
        'released_at': '2001-04-11',
        'mana_cost': '{U}{U}',
        'type_line': 'Instant',
        'oracle_text': 'Counter target spell.',
        'colors': ['U'],
        'color_identity': ['U'],
        'finishes': ['nonfoil'],
        'legalities': {'commander': 'legal'},
        'purchase_uris': {'tcgplayer': 'https://example.test/tcg/counterspell'},
        'tcgplayer_id': 333333,
        'cardmarket_id': 444444,
    },
]

identifiers_payload = {
    'data': {
        'uuid-1': {'identifiers': {'scryfallId': 's1', 'tcgplayerProductId': '534658', 'cardKingdomId': '12345', 'mcmId': '752712'}},
        'uuid-2': {'identifiers': {'scryfallId': 's2', 'tcgplayerProductId': '271111', 'cardKingdomFoilId': '54321', 'mcmId': '665555'}},
        'uuid-3': {'identifiers': {'scryfallId': 's3', 'tcgplayerProductId': '333333', 'cardKingdomId': '99999', 'mcmId': '444444'}},
    }
}

prices_payload = {
    'data': {
        'uuid-1': {
            'paper': {
                'tcgplayer': {
                    'currency': 'USD',
                    'retail': {'normal': {'2026-03-27': 2.92}},
                    'buylist': {'normal': {'2026-03-27': 1.10}},
                },
                'cardmarket': {
                    'currency': 'EUR',
                    'retail': {'normal': {'2026-03-27': 1.51}},
                },
            }
        },
        'uuid-2': {
            'paper': {
                'tcgplayer': {
                    'currency': 'USD',
                    'retail': {
                        'normal': {'2026-03-27': 1.75},
                        'foil': {'2026-03-27': 4.25},
                    },
                    'buylist': {
                        'normal': {'2026-03-27': 0.75},
                        'foil': {'2026-03-27': 2.00},
                    },
                }
            }
        },
        'uuid-3': {
            'paper': {
                'tcgplayer': {
                    'currency': 'USD',
                    'retail': {'normal': {'2026-03-27': 1.25}},
                    'buylist': {'normal': {'2026-03-27': 0.45}},
                }
            }
        },
    }
}

SCRYFALL_JSON.write_text(json.dumps(scryfall_payload), encoding='utf-8')
IDENTIFIERS_JSON.write_text(json.dumps(identifiers_payload), encoding='utf-8')
PRICES_JSON.write_text(json.dumps(prices_payload), encoding='utf-8')

print(SCRYFALL_JSON)
print(IDENTIFIERS_JSON)
print(PRICES_JSON)


## Run the Importers

The importer layer is split into three main jobs: Scryfall card rows,
MTGJSON identifier enrichment, and MTGJSON price snapshots.


In [ ]:
initialize_database(DB_PATH)

scryfall_stats = import_scryfall_cards(DB_PATH, SCRYFALL_JSON)
identifier_stats = import_mtgjson_identifiers(DB_PATH, IDENTIFIERS_JSON)
price_stats = import_mtgjson_prices(DB_PATH, PRICES_JSON)

print('scryfall:', scryfall_stats)
print('identifiers:', identifier_stats)
print('prices:', price_stats)


## Inspect the Imported Tables

`mtg_cards` is the printing catalog. `price_snapshots` stores provider,
finish, price kind, currency, and date slices for later valuation/reporting.


In [ ]:
with connect(DB_PATH) as connection:
    card_rows = [
        dict(row)
        for row in connection.execute(
            """
            SELECT scryfall_id, mtgjson_uuid, name, set_code, collector_number,
                   tcgplayer_product_id, cardkingdom_id, cardmarket_id
            FROM mtg_cards
            ORDER BY name
            """
        )
    ]
    price_rows = [
        dict(row)
        for row in connection.execute(
            """
            SELECT scryfall_id, provider, price_kind, finish, currency, snapshot_date, price_value
            FROM price_snapshots
            ORDER BY scryfall_id, provider, price_kind, finish
            """
        )
    ]
    price_summary_rows = [
        dict(row)
        for row in connection.execute(
            """
            SELECT provider, currency, finish, COUNT(*) AS rows,
                   MIN(snapshot_date) AS first_date,
                   MAX(snapshot_date) AS last_date
            FROM price_snapshots
            GROUP BY provider, currency, finish
            ORDER BY provider, currency, finish
            """
        )
    ]

print('Cards')
print(render_table(card_rows, [
    ('scryfall_id', 'scryfall_id'),
    ('mtgjson_uuid', 'uuid'),
    ('name', 'name'),
    ('set_code', 'set'),
    ('collector_number', 'number'),
    ('tcgplayer_product_id', 'tcgplayer'),
    ('cardkingdom_id', 'cardkingdom'),
    ('cardmarket_id', 'cardmarket'),
]))
print()
print('Price snapshots')
print(render_table(price_rows, [
    ('scryfall_id', 'scryfall_id'),
    ('provider', 'provider'),
    ('price_kind', 'kind'),
    ('finish', 'finish'),
    ('currency', 'currency'),
    ('snapshot_date', 'snapshot_date'),
    ('price_value', 'price_value'),
]))
print()
print('Price summary')
print(render_table(price_summary_rows, [
    ('provider', 'provider'),
    ('currency', 'currency'),
    ('finish', 'finish'),
    ('rows', 'rows'),
    ('first_date', 'first_date'),
    ('last_date', 'last_date'),
]))


The last table is the important contract check for this repo state:

- the EUR Cardmarket row from the sample payload was ignored
- finish values are already in the normalized vocabulary the inventory layer
  expects later (`normal`, `foil`, `etched`)
